# 02 — Exploratory Data Analysis (EDA)

Comprehensive visual analysis of Algeria's food price landscape:

- Price distribution by product and region
- Seasonal patterns (monthly, Ramadan)
- Regional price divergence
- Year-over-year inflation trends
- Correlation heatmap between products

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data_ingestion.wfp_connector import WFPConnector
from src.preprocessing.data_cleaner import DataCleaner
from src.utils.helpers import load_config

config = load_config('../config/config.yaml')
sns.set_theme(style='whitegrid', palette='muted')

# Load data
wfp = WFPConnector(config=config)
raw = wfp.fetch_price_data()
cleaner = DataCleaner(config=config)
df = cleaner.clean(raw)
print(f'Clean dataset: {df.shape}')
df.head()

In [ ]:
# ── 1. Price Distribution per Product ────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
order = df.groupby('product')['price'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='product', y='price', order=order, ax=ax, palette='Set2')
ax.set_title('Distribution des Prix par Produit (DZD)', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Prix (DZD)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2. Monthly Seasonality for Top Products ───────────────────
top_products = df.groupby('product')['price'].median().nlargest(4).index
seasonal = df[df['product'].isin(top_products)].copy()
seasonal['month'] = seasonal['date'].dt.month
monthly = seasonal.groupby(['product', 'month'])['price'].mean().reset_index()

fig = px.line(
    monthly, x='month', y='price', color='product',
    template='plotly_white', markers=True,
    title='Saisonnalité Mensuelle des Prix (Moyenne)',
    labels={'price': 'Prix Moyen (DZD)', 'month': 'Mois'}
)
fig.update_xaxes(tickvals=list(range(1,13)), ticktext=['Jan','Fév','Mar','Avr','Mai','Juin','Jul','Aoû','Sep','Oct','Nov','Déc'])
fig.show()

In [ ]:
# ── 3. Ramadan Price Effect ───────────────────────────────────
from src.utils.helpers import is_ramadan
df['is_ramadan'] = df['date'].apply(is_ramadan).astype(int)
ramadan_effect = (
    df.groupby(['product', 'is_ramadan'])['price']
    .mean()
    .unstack()
    .rename(columns={0: 'Hors Ramadan', 1: 'Ramadan'})
)
ramadan_effect['Effet (%)'] = (ramadan_effect['Ramadan'] / ramadan_effect['Hors Ramadan'] - 1) * 100
ramadan_effect.sort_values('Effet (%)', ascending=False, inplace=True)

fig, ax = plt.subplots(figsize=(10, 5))
ramadan_effect['Effet (%)'].plot(kind='barh', ax=ax, color=[
    '#d32f2f' if v > 0 else '#388e3c' for v in ramadan_effect['Effet (%)']
])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Effet Ramadan sur les Prix (%)', fontsize=13, fontweight='bold')
ax.set_xlabel('Variation (%)')
plt.tight_layout()
plt.show()
print(ramadan_effect)

In [ ]:
# ── 4. YoY Inflation ──────────────────────────────────────────
df['year'] = df['date'].dt.year
yearly = df.groupby(['year', 'product'])['price'].mean().unstack()
yoy = yearly.pct_change() * 100

fig = px.imshow(
    yoy.T,
    color_continuous_scale='RdYlGn_r',
    color_continuous_midpoint=0,
    title='Inflation Annuelle des Prix (%) — Heatmap',
    labels={'x': 'Année', 'y': 'Produit', 'color': 'Δ%'},
    template='plotly_white'
)
fig.show()

In [ ]:
# ── 5. Regional Price Divergence ─────────────────────────────
latest = df[df['date'] == df['date'].max()]
pivot_region = latest.pivot_table(values='price', index='product', columns='region', aggfunc='mean')

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    pivot_region.fillna(0), annot=True, fmt='.0f',
    cmap='YlOrRd', ax=ax, linewidths=0.3
)
ax.set_title('Prix Actuels par Produit × Région (DZD)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 6. Cross-product Correlation ─────────────────────────────
pivot_corr = df.pivot_table(values='price', index='date', columns='product', aggfunc='mean')
corr = pivot_corr.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, ax=ax, linewidths=0.3
)
ax.set_title('Corrélation entre Produits (Prix Mensuel)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()